In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

# Import warnings
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from __future__ import annotations

# ── PULLBACK: stacks that reverse after a strong move ─────────────────────────────────────────────────────────
#
# THE QUESTION (user, 2026-09-21): which stacks, after a STRONG MOVE with no reason behind it, give it back? E.g. a
# stack that climbs for no reason in the first half hour after 04:00 and then falls over the next half hour or hour
# is a good SHORT; the mirror image (a fall that is bought back) is a good LONG. "Different time points", so every
# anchor time of the day is scanned, not one.
#
# WHAT IS MEASURED
#   anchor T0 (every `anchor_step` minutes inside a zone)  ->  MOVE over M minutes: T0 -> T1 = T0 + M
#   the move must be STRONG:  |excess move| >= max(strong_k * scale, min_move_pp), where
#       excess move = stack move - beta * bench move   (beta and bench only if the data has them; else the raw move)
#       scale       = the stack's own typical M-minute move, taken from the PREVIOUS days only (no look-ahead)
#   then REVERSAL over R minutes: T1 -> T2 = T1 + R.
#   trade = fade the move at T1, close at T2:  short after an up move (entry at the bid, cover at the ask), long
#   after a down move (entry at the ask, sell at the bid). Without bid/ask in the data (final.parquet has only Stack%)
#   the same trade is the mid-to-mid reversal minus a flat `cost_pp`.
#   ONE event per (stack, day, zone, M, R, direction): the FIRST strong move of the zone that day, so overlapping
#   anchors are not counted as independent evidence.
#
# THE RULES ABOUT DROPPED PERIODS (same as the other strategies): rows inside `ignore_windows` (03:58-04:02 and
# 07:58-08:02) are dropped ALWAYS. A move or a reversal never starts or ends on a dropped print - the price at a
# time is the first valid print at or after it (within `price_tol` minutes), so an anchor at 04:00 reads 04:03.
#
# HOW IT IS JUDGED (`pullback_analyze`): sessions are split in time - the first part picks the stacks, the last part
# tests them; the same picking is also run on sign-shuffled events to say how many "stacks that reverse" pure luck
# would produce. A stack only counts if it holds up on the days it was NOT picked on.
import numpy as np
import pandas as pd

IGNORE_WINDOWS_DEFAULT = (((3, 58), (4, 2)), ((7, 58), (8, 2)))

# name, first anchor, last anchor, latest allowed end of the reversal (all hh:mm, NY clock)
ZONES_DEFAULT = (
    ("EARLY",     (4, 0),  (6, 0),   (9, 20)),
    ("PRE_MID",   (6, 0),  (8, 0),   (9, 20)),
    ("PRE_LATE",  (8, 0),  (9, 15),  (9, 28)),
    ("OPEN",      (9, 30), (10, 30), (16, 0)),
    ("MORNING",   (10, 30), (12, 0), (16, 0)),
    ("MIDDAY",    (12, 0), (14, 0),  (16, 0)),
    ("AFTERNOON", (14, 0), (15, 15), (16, 0)),
    ("POST",      (16, 0), (18, 30), (20, 0)),
)


def _m(hm):
    return int(hm[0]) * 60 + int(hm[1])


def default_cfg(**over):
    cfg = dict(
        zones=ZONES_DEFAULT,
        move_minutes=(15, 30, 60),
        reversal_minutes=(15, 30, 60, 90),
        anchor_step=15,
        price_tol=5,                     # a price at time t is the first valid print in [t, t + price_tol]
        ignore_windows=IGNORE_WINDOWS_DEFAULT,
        strong_k=2.5,                    # strong = |excess move| >= strong_k * the stack's own typical move ...
        min_move_pp=1.0,                 # ... and at least this many pp
        cost_pp=0.0,                     # flat round-trip cost, ONLY when the data has no bid/ask
        max_spread_pp=None,              # skip an event whose entry or exit spread is wider than this (pp of the close)
        exclude_news=True,               # skip stacks with news that day, when the data says so
        scale_days=10, min_scale_days=3,  # scale = median of the last `scale_days` days' typical move
        max_abs_pp=40.0,                 # a print beyond this is a broken quote, not a price
    )
    cfg.update(over)
    return cfg


# ── one session day as a matrix ─────────────────────────────────────────────────────────────────────────
def make_panel(tickers, t_first, bid, ask, bench_mid=None, beta=None, news=None):
    """bid/ask: float32 [N tickers, T minutes] in % of the last close (Stack%-style); column j is minute t_first + j.
    bench_mid: same shape, the benchmark's % move (optional). beta: [N] (optional). news: bool [N] (optional)."""
    return dict(tickers=np.asarray(tickers), t_first=int(t_first), bid=np.asarray(bid, np.float32), ask=np.asarray(ask, np.float32),
                bench=None if bench_mid is None else np.asarray(bench_mid, np.float32),
                beta=None if beta is None else np.asarray(beta, np.float32), news=None if news is None else np.asarray(news, bool))


def panel_from_stack(tickers, t_first, stack, bench_pct=None, beta=None):
    """final.parquet-style data: only one price per minute, so bid = ask = the stack."""
    s = np.asarray(stack, np.float32)
    return make_panel(tickers, t_first, s, s, bench_pct, beta)


def _clean(panel, cfg):
    """NaN out the dropped windows, crossed / absurd quotes; return bid, ask, mid, bench."""
    t_first = panel["t_first"]
    bid, ask = panel["bid"].copy(), panel["ask"].copy()
    T = bid.shape[1]
    bad = ~(np.isfinite(bid) & np.isfinite(ask)) | (ask < bid) | (np.abs(bid) > cfg["max_abs_pp"]) | (np.abs(ask) > cfg["max_abs_pp"])
    bid[bad] = np.nan
    ask[bad] = np.nan
    bench = None if panel["bench"] is None else panel["bench"].copy()
    for a, b in cfg["ignore_windows"]:
        lo, hi = _m(a) - t_first, _m(b) - t_first
        lo, hi = max(lo, 0), min(hi, T - 1)
        if hi >= lo:
            bid[:, lo:hi + 1] = np.nan
            ask[:, lo:hi + 1] = np.nan
            if bench is not None:
                bench[:, lo:hi + 1] = np.nan
    return bid, ask, (bid + ask) * 0.5, bench


def _next_valid(x):
    """index of the first finite value at or after each column (BIG when none), per row."""
    N, T = x.shape
    idx = np.where(np.isfinite(x), np.arange(T, dtype=np.int32)[None, :], np.int32(1 << 20))
    return np.minimum.accumulate(idx[:, ::-1], axis=1)[:, ::-1]


class _Px:
    """price lookup: the first valid print at or after minute t, within tol; NaN otherwise."""

    def __init__(self, arrays, t_first, tol):
        self.t_first, self.tol = t_first, tol
        self.arrays = arrays
        self.nxt = _next_valid(arrays[0])
        self.T = arrays[0].shape[1]
        self.rows = np.arange(arrays[0].shape[0])

    def at(self, t):
        j = t - self.t_first
        if j < 0 or j >= self.T:
            return [np.full(self.rows.size, np.nan, np.float32) for _ in self.arrays], np.full(self.rows.size, -1)
        jj = self.nxt[:, j]
        ok = (jj - j) <= self.tol
        jj = np.where(ok, jj, 0)
        out = [np.where(ok, a[self.rows, jj], np.nan).astype(np.float32) for a in self.arrays]
        return out, np.where(ok, jj + self.t_first, -1)


def day_scales(panel, cfg):
    """the stack's typical M-minute move on this day (robust: 1.4826 x MAD of the M-minute change, sampled every 5 min)"""
    bid, ask, mid, _ = _clean(panel, cfg)
    px = _Px([mid], panel["t_first"], cfg["price_tol"])
    lo, hi = max(_m((4, 3)), panel["t_first"]), min(_m((16, 0)), panel["t_first"] + mid.shape[1] - 1)
    out = {}
    for M in cfg["move_minutes"]:
        diffs = []
        for t in range(lo, hi - M + 1, 5):
            a, _ = px.at(t)
            b, _ = px.at(t + M)
            diffs.append(b[0] - a[0])
        if not diffs:
            out[M] = np.full(mid.shape[0], np.nan, np.float32)
            continue
        d = np.stack(diffs, axis=1)
        with np.errstate(all="ignore"):
            med = np.nanmedian(d, axis=1)
            mad = np.nanmedian(np.abs(d - med[:, None]), axis=1)
        cnt = np.isfinite(d).sum(axis=1)
        s = (1.4826 * mad).astype(np.float32)
        s[cnt < 20] = np.nan
        out[M] = s
    return out


def scan_day(panel, cfg, scale):
    """-> DataFrame of events for one day. `scale`: {M: float[N]} the stacks' typical M-minute moves from PREVIOUS days."""
    bid, ask, mid, bench = _clean(panel, cfg)
    t_first = panel["t_first"]
    N, T = mid.shape
    tk = panel["tickers"]
    beta = panel["beta"] if panel["beta"] is not None else np.ones(N, np.float32)
    beta = np.where(np.isfinite(beta), beta, 1.0).astype(np.float32)
    arrays = [mid, bid, ask] + ([bench] if bench is not None else [])
    px = _Px(arrays, t_first, cfg["price_tol"])
    news = panel["news"] if (panel["news"] is not None and cfg["exclude_news"]) else np.zeros(N, bool)
    has_book = bool((np.nan_to_num(ask - bid) > 0).any())
    rows = []
    last_t = t_first + T - 1
    for zi, (zname, zfrom, zto, zlimit) in enumerate(cfg["zones"]):
        z0, z1, zl = _m(zfrom), _m(zto), _m(zlimit)
        for M in cfg["move_minutes"]:
            sc = scale.get(M)
            for R in cfg["reversal_minutes"]:
                taken = {1: news.copy(), -1: news.copy()}
                for t0 in range(z0, z1 + 1, cfg["anchor_step"]):
                    t1, t2 = t0 + M, t0 + M + R
                    if t2 > zl or t2 > last_t:
                        break
                    v0, _ = px.at(t0)
                    v1, s1 = px.at(t1)
                    v2, s2 = px.at(t2)
                    move = v1[0] - v0[0]
                    if bench is not None:
                        move = move - beta * (v1[3] - v0[3])
                    thr = np.full(N, cfg["min_move_pp"], np.float32)
                    if sc is not None:
                        thr = np.maximum(thr, cfg["strong_k"] * np.nan_to_num(sc, nan=np.inf))
                    strong = np.isfinite(move) & (np.abs(move) >= thr) & np.isfinite(v2[0]) & (s1 > 0) & (s2 > 0)
                    for d in (1, -1):
                        cand = strong & (np.sign(move) == d) & ~taken[d]
                        if not cand.any():
                            continue
                        if d > 0:      # up move -> short: sell at the bid at T1, buy back at the ask at T2
                            gross = v1[0] - v2[0]
                            pnl = v1[1] - v2[2]
                            spr1, spr2 = v1[2] - v1[1], v2[2] - v2[1]
                        else:          # down move -> long: buy at the ask at T1, sell at the bid at T2
                            gross = v2[0] - v1[0]
                            pnl = v2[1] - v1[2]
                            spr1, spr2 = v1[2] - v1[1], v2[2] - v2[1]
                        if not has_book:
                            pnl = gross - cfg["cost_pp"]
                        if cfg["max_spread_pp"] is not None and has_book:
                            cand &= (spr1 <= cfg["max_spread_pp"]) & (spr2 <= cfg["max_spread_pp"])
                        cand &= np.isfinite(pnl)
                        if not cand.any():
                            continue
                        ii = np.flatnonzero(cand)
                        taken[d][ii] = True
                        rows.append(pd.DataFrame({
                            "ticker": tk[ii], "zone": zname, "M": M, "R": R, "dir": d, "t0": t0,
                            "move": move[ii], "gross": gross[ii], "net": pnl[ii],
                            "spr1": (spr1[ii] if has_book else 0.0), "spr2": (spr2[ii] if has_book else 0.0),
                        }))
    if not rows:
        return pd.DataFrame(columns=["ticker", "zone", "M", "R", "dir", "t0", "move", "gross", "net", "spr1", "spr2"])
    return pd.concat(rows, ignore_index=True)


def scan_days(day_iter, cfg):
    """day_iter yields (sdate, panel) in date order. Returns the events of every day that had enough history for a scale."""
    hist = []          # per previous day: {M: scale array}, aligned to that day's tickers
    hist_tk = []
    frames = []
    for sdate, panel in day_iter:
        scale = {}
        if len(hist) >= cfg["min_scale_days"]:
            for M in cfg["move_minutes"]:
                acc = {}
                for tks, sc in zip(hist_tk[-cfg["scale_days"]:], hist[-cfg["scale_days"]:]):
                    for t_, v_ in zip(tks, sc[M]):
                        if np.isfinite(v_):
                            acc.setdefault(t_, []).append(v_)
                cur = np.full(len(panel["tickers"]), np.nan, np.float32)
                for i, t_ in enumerate(panel["tickers"]):
                    vs = acc.get(t_)
                    if vs:
                        cur[i] = float(np.median(vs))
                scale[M] = cur
            ev = scan_day(panel, cfg, scale)
            if len(ev):
                ev.insert(0, "sdate", int(sdate))
                frames.append(ev)
        hist.append(day_scales(panel, cfg))
        hist_tk.append(panel["tickers"])
    if not frames:
        return pd.DataFrame(columns=["sdate", "ticker", "zone", "M", "R", "dir", "t0", "move", "gross", "net", "spr1", "spr2"])
    return pd.concat(frames, ignore_index=True)


In [ ]:
from __future__ import annotations

# ── stage 1: final.parquet -> one small parquet per SESSION DAY ────────────────────────────────────────────
#
# final.parquet is sorted by ticker; a day-by-day scan needs every ticker of one day at once. One sequential pass
# writes [ticker, smin, stack, bench] per session date; stage 2 (`pullback_days`) pivots one day at a time, so memory
# is one day, not the file. Session days follow PairFlux: rows at/after `session_split_min` (17:00) belong to the NEXT
# session day and are numbered negative (21:00 -> -180), so a session is one increasing axis from -180 to 20:00.
import numpy as np
import pandas as pd


def pullback_stage_by_day(input_path, stage_dir, *, session_split_min=1020, start_date=None,
                          STOCK_NUM_FIELD="Stack%", BENCH_NUM_FIELD="Bench%", smin_lo=-180, smin_hi=1199,
                          log_every_n_chunks=20):
    import gc, time, shutil
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    start_i = int(start_date.replace("-", "")) if start_date else -1

    schema = pa.schema([("ticker", pa.string()), ("smin", pa.int16()), ("stack", pa.float32()), ("bench", pa.float32())])
    writers, counts = {}, {}

    def _writer(sdate):
        if sdate not in writers:
            writers[sdate] = pq.ParquetWriter(str(stage / f"{sdate}.parquet"), schema, compression="zstd")
            counts[sdate] = 0
        return writers[sdate]

    pf = pq.ParquetFile(input_path)
    want = ["ticker", "dt", STOCK_NUM_FIELD, BENCH_NUM_FIELD]
    cols = [c for c in want if c in pf.schema.names]
    for need in ("ticker", "dt", STOCK_NUM_FIELD):
        if need not in cols:
            raise KeyError(f"final.parquet is missing required column: {need}")
    has_bench = BENCH_NUM_FIELD in cols
    t0 = time.time()
    total_in = total_out = 0
    print(f"START pullback stage1  file={input_path}  bench column={'yes' if has_bench else 'no (raw moves)'}")
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            t_arr = dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 + dt.dt.minute.to_numpy(dtype="int32", copy=False)
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 1440, t_arr).astype("int16")
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 + sess.dt.month.to_numpy(dtype="int32", copy=False) * 100
                     + sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")
            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            bench = (pd.to_numeric(df[BENCH_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
                     if has_bench else np.full(len(df), np.nan, np.float32))
            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue
            out = pd.DataFrame({"ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str), "sdate": sdate[keep],
                                "smin": smin[keep], "stack": stack[keep], "bench": bench[keep]})
            for sd, part in out.groupby("sdate", sort=False):
                _writer(int(sd)).write_table(pa.Table.from_pandas(part[["ticker", "smin", "stack", "bench"]], schema=schema, preserve_index=False))
                counts[int(sd)] += len(part)
                total_out += len(part)
            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci + 1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} days={len(writers)} elapsed={time.time() - t0:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()
    print(f"DONE stage1 in={total_in:,} staged={total_out:,} days={len(writers)} elapsed={time.time() - t0:.1f}s")
    return sorted(counts)


def pullback_days(stage_dir, *, smin_lo=-180, smin_hi=1199, min_tickers=30):
    """yields (sdate, panel) in date order, one pivoted day at a time."""
    from pathlib import Path
    for f in sorted(Path(stage_dir).glob("*.parquet"), key=lambda p: int(p.stem)):
        df = pd.read_parquet(f)
        if df["ticker"].nunique() < min_tickers:
            continue
        # a stack printed twice in one minute: keep the last print
        df = df.drop_duplicates(["ticker", "smin"], keep="last")
        tick, ti = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        T = smin_hi - smin_lo + 1
        stack = np.full((tick.size, T), np.nan, np.float32)
        bench = np.full((tick.size, T), np.nan, np.float32)
        j = df["smin"].to_numpy().astype(np.int64) - smin_lo
        stack[ti, j] = df["stack"].to_numpy()
        bench[ti, j] = df["bench"].to_numpy()
        has_bench = bool(np.isfinite(bench).any())
        yield int(f.stem), panel_from_stack(tick, smin_lo, stack, bench if has_bench else None)


In [ ]:
from __future__ import annotations

# ── analysis: which stacks reverse, and do they keep doing it on days they were not picked on? ──────────────
import numpy as np
import pandas as pd

KEY = ["ticker", "zone", "M", "R", "dir"]


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _agg(ev):
    g = ev.groupby(KEY, sort=False)
    out = g["net"].agg(n="size", mean_net="mean", sd_net="std").reset_index()
    out["hits"] = g["net"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out["mean_gross"] = g["gross"].mean().to_numpy()
    out["mean_move"] = g["move"].apply(lambda s: float(np.abs(s).mean())).to_numpy()
    with np.errstate(all="ignore"):
        out["t"] = out["mean_net"] / (out["sd_net"] / np.sqrt(out["n"]))
    out["hit_rate"] = out["hits"] / out["n"]
    out["hit_lb"] = _wilson_lb(out["hits"], out["n"])
    return out


def _select(tr, min_n, min_hit_lb, min_t, min_mean):
    return tr[(tr["n"] >= min_n) & (tr["hit_lb"] >= min_hit_lb) & (tr["t"] >= min_t) & (tr["mean_net"] >= min_mean)]


def pullback_analyze(events, *, train_share=2 / 3, min_n=6, min_hit_lb=0.5, min_t=2.0, min_mean_pp=0.3,
                     test_min_n=2, null_rounds=5, seed=7, verbose=True):
    """events: DataFrame from scan_days. Returns a dict of DataFrames + a summary dict."""
    if events.empty:
        return dict(selected=events, pooled=events, summary={"events": 0})
    days = np.array(sorted(events["sdate"].unique()))
    cut = days[max(1, int(round(len(days) * train_share))) - 1]
    train, test = events[events["sdate"] <= cut], events[events["sdate"] > cut]
    tr, te = _agg(train), _agg(test)
    sel = _select(tr, min_n, min_hit_lb, min_t, min_mean_pp)
    te = te.add_prefix("test_").rename(columns={f"test_{k}": k for k in KEY})
    sel = sel.merge(te, on=KEY, how="left")

    # pooled baseline: EVERY stack, per cell - is there a general pullback in this zone at all?
    pooled = events.groupby(["zone", "M", "R", "dir"], sort=False).agg(
        events=("net", "size"), mean_gross=("gross", "mean"), mean_net=("net", "mean"),
        hit_rate=("net", lambda s: float((s > 0).mean())), stacks=("ticker", "nunique")).reset_index()

    # how many "stacks that reverse" would luck alone give? The same picking, run after the TICKER labels are shuffled
    # among the events of every (zone, M, R, direction) cell: each stack keeps its number of events and the outcomes
    # keep their real (skewed, mostly losing) distribution, but nothing belongs to a stack any more.
    rng = np.random.default_rng(seed)
    null_counts = []
    for _ in range(null_rounds):
        sh = train.copy()
        sh["ticker"] = sh.groupby(["zone", "M", "R", "dir"], sort=False)["ticker"].transform(lambda s: rng.permutation(s.to_numpy()))
        null_counts.append(len(_select(_agg(sh), min_n, min_hit_lb, min_t, min_mean_pp)))

    ok = sel[(sel["test_n"] >= test_min_n)]
    summary = dict(
        events=int(len(events)), days=int(len(days)), train_days=int((days <= cut).sum()), test_days=int((days > cut).sum()),
        split_after=int(cut), ticker_cells_scored=int(len(tr)), picked_on_train=int(len(sel)),
        picked_by_luck_avg=float(np.mean(null_counts)) if null_counts else None,
        picked_with_test_events=int(len(ok)),
        test_mean_net_pp=float(np.average(ok["test_mean_net"], weights=ok["test_n"])) if len(ok) else None,
        test_hit_rate=float(np.average(ok["test_hit_rate"], weights=ok["test_n"])) if len(ok) else None,
        train_mean_net_pp=float(np.average(ok["mean_net"], weights=ok["n"])) if len(ok) else None,
        stacks_picked=int(sel["ticker"].nunique()),
        stacks_confirmed=int(ok[(ok["test_mean_net"] > 0)]["ticker"].nunique()) if len(ok) else 0,
    )
    if verbose:
        for k, v in summary.items():
            print(f"  {k:<26} {v}")
    return dict(selected=sel.sort_values(["t"], ascending=False), pooled=pooled, summary=summary, train=tr)


In [ ]:
from pathlib import Path
import os, json, gzip


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# ── runner ───────────────────────────────────────────────────────────────────────────────────────────────
FINAL_PATH, OUT_DIR = _resolve_orion_paths("pullback")

# the per-day stage lives OUTSIDE signals/ (the publisher pushes the whole signals/ tree - see PairFlux.ipynb)
STAGE_DIR = OUT_DIR.parent.parent / "_work" / "pullback_stage"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)

_stage_ok = STAGE_DIR.exists() and any(STAGE_DIR.glob("*.parquet")) and \
    min(p.stat().st_mtime for p in STAGE_DIR.glob("*.parquet")) >= FINAL_PATH.stat().st_mtime
if _stage_ok:
    print("stage is at least as new as final.parquet - reusing", STAGE_DIR)
else:
    pullback_stage_by_day(str(FINAL_PATH), str(STAGE_DIR), session_split_min=1020)

CFG = default_cfg(
    # final.parquet has one price per minute (Stack%), no bid/ask: a round trip is charged this flat cost. Pre-market
    # quotes are WIDE, so treat a result with a small cost as an upper bound, not a promise.
    cost_pp=0.3,
    strong_k=2.5, min_move_pp=1.0,
)

events = scan_days(pullback_days(str(STAGE_DIR)), CFG)
print(f"events: {len(events):,} over {events['sdate'].nunique() if len(events) else 0} days")

res = pullback_analyze(events, train_share=2 / 3, min_n=6, min_hit_lb=0.5, min_t=2.0, min_mean_pp=0.3)

sel, pooled = res["selected"], res["pooled"]
sel.to_csv(OUT_DIR / "pullback_stacks.csv", index=False)
pooled.to_csv(OUT_DIR / "pullback_pooled.csv", index=False)
with open(OUT_DIR / "pullback_meta.json", "w", encoding="utf-8") as f:
    json.dump({"summary": res["summary"], "cfg": {k: (list(map(list, v)) if k in ("zones", "ignore_windows") else v) for k, v in CFG.items()}},
              f, ensure_ascii=False, indent=1, default=str)
print("written ->", OUT_DIR)
